# GB News Politics Corpus (Wayback) — 2014–2024

This notebook:
1) Discovers GB News politics URLs using Wayback prefix search and/or GB News sitemaps
2) Queries Wayback CDX for captures within a year range (default 2014–2024)
3) Downloads archived HTML for a selected capture
4) Extracts main article text with trafilatura
5) Saves raw HTML + JSONL records (resume-safe)

Note: GB News is newer than 2014, so early-year coverage may be sparse.

In [1]:
import hashlib, json, time, random, re
from pathlib import Path
from typing import List, Dict, Optional, Tuple

import requests
from lxml import etree
from bs4 import BeautifulSoup
from tenacity import retry, wait_exponential, stop_after_attempt
from tqdm.auto import tqdm

from trafilatura import extract as traf_extract

/Users/ameliemajor/miniforge3/envs/gbnews/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ===========
# CONFIG
# ===========

FROM_YEAR = 2014
TO_YEAR = 2024

# Restrict to politics section URLs:
POLITICS_PREFIX = "https://www.gbnews.com/politics/"

# Output folders
BASE = Path("gbnews_corpus")
DATA = BASE / "data"
RAW = DATA / "raw_html"
OUT = DATA / "extracted_jsonl"
CACHE = BASE / "cache"
CDX_CACHE = CACHE / "cdx"
URL_CACHE = CACHE / "url_lists"
LOGS = BASE / "logs"

for p in [RAW, OUT, CDX_CACHE, URL_CACHE, LOGS]:
    p.mkdir(parents=True, exist_ok=True)

OUT_JSONL = OUT / f"gbnews_politics_{FROM_YEAR}_{TO_YEAR}.jsonl"

UA = {"User-Agent": "AcademicResearchBot/1.0 (contact: your-email)"}

# Rate limiting
BASE_SLEEP = 1.0     # seconds
JITTER = 0.5         # add 0..0.5 seconds

# Capture selection: "earliest" preserves publication proximity; "latest" can extract cleaner
CAPTURE_POLICY = "earliest"   # or "latest"

In [3]:
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

session = requests.Session()
session.headers.update(UA)

retries = Retry(
    total=5,
    backoff_factor=0.5,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET", "HEAD"],
    raise_on_status=False,
)
adapter = HTTPAdapter(max_retries=retries, pool_connections=20, pool_maxsize=20)
session.mount("https://", adapter)
session.mount("http://", adapter)

def polite_sleep(base: float = BASE_SLEEP):
    time.sleep(base + random.random() * JITTER)

@retry(wait=wait_exponential(min=1, max=30), stop=stop_after_attempt(5))
def get(url: str, params=None, timeout: Tuple[int, int] = (10, 60)) -> requests.Response:
    # timeout=(connect_timeout, read_timeout)
    r = session.get(url, params=params, timeout=timeout)
    r.raise_for_status()
    return r

In [4]:
def sha1(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8")).hexdigest()

def is_politics_url(url: str) -> bool:
    return url.startswith(POLITICS_PREFIX)

def load_done_urls(jsonl_path: Path) -> set[str]:
    done = set()
    if not jsonl_path.exists():
        return done
    with jsonl_path.open("r", encoding="utf-8") as f:
        for line in f:
            try:
                obj = json.loads(line)
                if "original_url" in obj:
                    done.add(obj["original_url"])
            except Exception:
                pass
    return done

In [5]:
def parse_sitemap(sitemap_url: str, verbose: bool = False) -> List[str]:
    if verbose:
        print(f"🔎 Fetching sitemap: {sitemap_url}")
    xml = get(sitemap_url).content

    root = etree.fromstring(xml)
    ns = root.nsmap.get(None, "")
    nsmap = {"ns": ns} if ns else None

    # sitemap index
    if root.tag.endswith("sitemapindex"):
        locs = (
            root.xpath("//ns:sitemap/ns:loc/text()", namespaces=nsmap)
            if nsmap else
            root.xpath("//sitemap/loc/text()")
        )
        urls: List[str] = []
        for loc in locs:
            urls.extend(parse_sitemap(loc, verbose=verbose))
        return urls

    # urlset
    locs = (
        root.xpath("//ns:url/ns:loc/text()", namespaces=nsmap)
        if nsmap else
        root.xpath("//url/loc/text()")
    )
    return [u.strip() for u in locs]

def list_numbered_sitemaps_from_index(index_url: str = "https://www.gbnews.com/sitemap.xml") -> List[str]:
    idx = parse_sitemap(index_url, verbose=False)
    numbered = [u for u in idx if "/feeds/sitemaps/sitemap_" in u and u.endswith(".xml")]
    return numbered

In [6]:
def wayback_list_urls_for_prefix(prefix: str, from_year: int, to_year: int, limit: int = 50000) -> List[str]:
    """
    Returns unique original URLs archived under a prefix during a year range.
    """
    cache_key = sha1(f"{prefix}|{from_year}|{to_year}|{limit}")
    cache_path = URL_CACHE / f"prefix_{cache_key}.json"

    if cache_path.exists():
        return json.loads(cache_path.read_text(encoding="utf-8"))

    cdx_url = "https://web.archive.org/cdx/search/cdx"
    params = {
        "url": prefix + "*",
        "matchType": "prefix",
        "output": "json",
        "from": str(from_year),
        "to": str(to_year),
        "filter": "statuscode:200",
        "collapse": "urlkey",   # unique URLs
        "fl": "original",
        "limit": str(limit),
    }
    j = get(cdx_url, params=params).json()
    if len(j) <= 1:
        urls = []
    else:
        urls = [row[0] for row in j[1:]]

    cache_path.write_text(json.dumps(urls, ensure_ascii=False, indent=2), encoding="utf-8")
    return urls

In [7]:
def cdx_cache_path(url: str, from_year: int, to_year: int) -> Path:
    return CDX_CACHE / f"{sha1(url)}_{from_year}_{to_year}.json"

def wayback_cdx(url: str, from_year: int, to_year: int) -> List[Dict]:
    """
    List captures for a URL within a year range, deduped by digest.
    """
    cache_file = cdx_cache_path(url, from_year, to_year)
    if cache_file.exists():
        j = json.loads(cache_file.read_text(encoding="utf-8"))
    else:
        cdx_url = "https://web.archive.org/cdx/search/cdx"
        params = {
            "url": url,
            "output": "json",
            "from": str(from_year),
            "to": str(to_year),
            "filter": "statuscode:200",
            "collapse": "digest",
            "fl": "timestamp,original,statuscode,mimetype,digest,length"
        }
        j = get(cdx_url, params=params).json()
        cache_file.write_text(json.dumps(j, ensure_ascii=False), encoding="utf-8")

    if len(j) <= 1:
        return []
    header = j[0]
    return [dict(zip(header, row)) for row in j[1:]]

def choose_capture(captures: List[Dict], policy: str = "earliest") -> Optional[Dict]:
    if not captures:
        return None
    return captures[0] if policy == "earliest" else captures[-1]

def fetch_snapshot(original_url: str, timestamp: str) -> Tuple[str, str]:
    snap_url = f"https://web.archive.org/web/{timestamp}/{original_url}"
    html = get(snap_url).text
    return snap_url, html

In [8]:
def extract_text(html: str) -> Dict:
    text = traf_extract(html, include_comments=False, include_tables=False)
    if text and len(text) > 200:
        return {"text": text, "method": "trafilatura"}

    # fallback: quick-and-dirty
    soup = BeautifulSoup(html, "html.parser")
    title = (soup.title.get_text(" ", strip=True) if soup.title else None)
    body = soup.get_text("\n", strip=True)
    return {"text": body if len(body) > 200 else None, "title": title, "method": "bs4_fallback"}

In [9]:
def process_urls(urls: List[str],
                 from_year: int,
                 to_year: int,
                 out_jsonl: Path,
                 total_limit: Optional[int] = None) -> Dict[str, int]:

    done = load_done_urls(out_jsonl)
    print(f"🔁 Resuming: {len(done)} URLs already saved in {out_jsonl.name}")

    stats = {"success": 0, "no_captures": 0, "no_text": 0, "errors": 0, "skipped_done": 0}

    with out_jsonl.open("a", encoding="utf-8") as f:
        for url in tqdm(urls, desc="Processing URLs"):
            if url in done:
                stats["skipped_done"] += 1
                continue

            try:
                captures = wayback_cdx(url, from_year, to_year)
                if not captures:
                    stats["no_captures"] += 1
                    polite_sleep()
                    continue

                cap = choose_capture(captures, policy=CAPTURE_POLICY)
                if not cap:
                    stats["no_captures"] += 1
                    polite_sleep()
                    continue

                ts = cap["timestamp"]
                snap_url, html = fetch_snapshot(url, ts)

                key = sha1(f"{url}|{ts}")
                (RAW / f"{key}.html").write_text(html, encoding="utf-8")

                extracted = extract_text(html)
                if not extracted.get("text"):
                    stats["no_text"] += 1
                    polite_sleep()
                    continue

                rec = {
                    "source": "gbnews",
                    "section_filter": "politics",
                    "original_url": url,
                    "wayback_timestamp": ts,
                    "wayback_url": snap_url,
                    "capture_policy": CAPTURE_POLICY,
                    "year_range": {"from": from_year, "to": to_year},
                    "extraction": extracted,
                }
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                done.add(url)
                stats["success"] += 1

                polite_sleep()

                if total_limit and stats["success"] >= total_limit:
                    print("🛑 total_limit reached")
                    break

            except KeyboardInterrupt:
                print("\n🛑 Interrupted by user. Progress saved.")
                break
            except Exception as e:
                stats["errors"] += 1
                # log lightweight error
                (LOGS / "errors.log").open("a", encoding="utf-8").write(f"{url}\t{type(e).__name__}\t{e}\n")
                polite_sleep()

    return stats

In [10]:
urls = wayback_list_urls_for_prefix(POLITICS_PREFIX, FROM_YEAR, TO_YEAR, limit=50000)
print("Wayback-discovered politics URLs:", len(urls))

# Safety: enforce the prefix filter (should already be true, but keep it clean)
urls = [u for u in urls if is_politics_url(u)]
print("After prefix filter:", len(urls))

# Optional: shuffle so you sample across the archive rather than clustered ordering
random.shuffle(urls)

# Peek
urls[:10]

Wayback-discovered politics URLs: 0
After prefix filter: 0


[]

In [11]:
stats = process_urls(
    urls=urls,
    from_year=FROM_YEAR,
    to_year=TO_YEAR,
    out_jsonl=OUT_JSONL,
    total_limit=200,   # increase/remove once you're happy
)
stats

🔁 Resuming: 0 URLs already saved in gbnews_politics_2014_2024.jsonl


Processing URLs: 0it [00:00, ?it/s]


{'success': 0, 'no_captures': 0, 'no_text': 0, 'errors': 0, 'skipped_done': 0}

In [12]:
print("Output JSONL:", OUT_JSONL)
print("Lines:", sum(1 for _ in OUT_JSONL.open("r", encoding="utf-8")) if OUT_JSONL.exists() else 0)

# show one record
if OUT_JSONL.exists():
    with OUT_JSONL.open("r", encoding="utf-8") as f:
        first = json.loads(next(f))
    first.keys(), first["original_url"], first["wayback_timestamp"], len(first["extraction"]["text"])

Output JSONL: gbnews_corpus/data/extracted_jsonl/gbnews_politics_2014_2024.jsonl
Lines: 0


StopIteration: 